# Actividad 6 — Fase 2: evaluación de riesgo crediticio
Este notebook descarga el dataset UCI, audita sus datos, compara DummyClassifier, Regresión Logística y Random Forest, y produce métricas, matrices de confusión y curvas ROC. **Los resultados solo existen después de ejecutar todas las celdas.**

In [ ]:
%pip -q install ucimlrepo pandas numpy scikit-learn matplotlib seaborn

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay)
SEED = 42
# nbconvert ejecuta normalmente desde notebooks/; conservar resultados en la raiz del repositorio.
OUTPUT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "results"
OUTPUT.mkdir(parents=True, exist_ok=True)

In [ ]:
dataset = fetch_ucirepo(id=350)
X = dataset.data.features.copy()
y = dataset.data.targets.iloc[:, 0].astype(int)
X = X.drop(columns=["ID"], errors="ignore")
assert len(X) == len(y) and set(y.unique()) == {0, 1}
print("Dimensiones:", X.shape)
print("Distribución de clases:", y.value_counts().sort_index().to_dict())
print("Nulos:", int(X.isna().sum().sum()), "Duplicados:", int(X.duplicated().sum()))
display(X.head())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=SEED)
# Las columnas categóricas son códigos y deben tratarse como categorías, no como magnitudes continuas.
cat_candidates = ["SEX", "EDUCATION", "MARRIAGE"]
cat_cols = [c for c in cat_candidates if c in X.columns]
num_cols = [c for c in X.columns if c not in cat_cols]
preprocess = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder(handle_unknown="ignore"))]), cat_cols)
])
models = {
    "Baseline": DummyClassifier(strategy="prior"),
    "Regresion_Logistica": LogisticRegression(max_iter=1500, random_state=SEED),
    "Random_Forest": RandomForestClassifier(n_estimators=120, min_samples_leaf=3, n_jobs=-1, random_state=SEED)
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
print("Train:", X_train.shape, "Test:", X_test.shape)

In [ ]:
cv_rows = []
for name, model in models.items():
    pipe = Pipeline([("preprocess", preprocess), ("model", model)])
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring={"roc_auc":"roc_auc", "f1":"f1", "recall":"recall", "accuracy":"accuracy"}, n_jobs=1)
    cv_rows.append({"modelo": name, **{f"{m}_media": float(scores[f"test_{m}"].mean()) for m in ["roc_auc","f1","recall","accuracy"]}, **{f"{m}_std": float(scores[f"test_{m}"].std()) for m in ["roc_auc","f1","recall","accuracy"]}})
cv_df = pd.DataFrame(cv_rows).sort_values("roc_auc_media", ascending=False)
display(cv_df)
cv_df.to_csv(OUTPUT / "validacion_cruzada.csv", index=False)

In [ ]:
# Se reporta un umbral fijo de 0.5 para esta fase. Su optimización se realizará en la Fase 3 SOLO con datos de desarrollo.
rows = []
fig, ax = plt.subplots(figsize=(7, 6))
for name, model in models.items():
    pipe = Pipeline([("preprocess", preprocess), ("model", model)])
    pipe.fit(X_train, y_train)
    scores = pipe.predict_proba(X_test)[:, 1]
    pred = (scores >= 0.5).astype(int)
    cm = confusion_matrix(y_test, pred, labels=[0,1])
    rows.append({"modelo":name, "accuracy":accuracy_score(y_test,pred), "precision":precision_score(y_test,pred,zero_division=0), "recall":recall_score(y_test,pred,zero_division=0), "f1":f1_score(y_test,pred,zero_division=0), "roc_auc":roc_auc_score(y_test,scores), "TN":int(cm[0,0]),"FP":int(cm[0,1]),"FN":int(cm[1,0]),"TP":int(cm[1,1])})
    fpr,tpr,_ = roc_curve(y_test,scores)
    ax.plot(fpr,tpr,label=f"{name} (AUC={roc_auc_score(y_test,scores):.3f})")
    fig_cm, ax_cm = plt.subplots(figsize=(5,4))
    ConfusionMatrixDisplay(cm,display_labels=["Cumple","Incumple"]).plot(ax=ax_cm,cmap="Blues",colorbar=False)
    ax_cm.set_title(f"Matriz de confusión — {name}")
    fig_cm.tight_layout()
    fig_cm.savefig(OUTPUT / f"matriz_{name.lower()}.png",dpi=150)
    plt.close(fig_cm)
ax.plot([0,1],[0,1],"k--",label="Azar")
ax.set(xlabel="Tasa de falsos positivos",ylabel="Tasa de verdaderos positivos",title="Curvas ROC — conjunto de prueba")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT / "curvas_roc.png",dpi=150)
plt.show()
metrics = pd.DataFrame(rows).sort_values("roc_auc",ascending=False)
display(metrics)
metrics.to_csv(OUTPUT / "metricas_test.csv",index=False)
print("Archivos generados en:", OUTPUT.resolve())

## Interpretación y limitaciones
- **Recall:** proporción de incumplimientos identificados.
- **Precision:** proporción de alertas de incumplimiento que eran correctas.
- **ROC-AUC:** capacidad de ordenar casos positivos por encima de negativos, independiente del umbral fijo.
- **FN:** incumplimientos no detectados; **FP:** clientes cumplidores señalados erróneamente.
- **Precaución:** esta fase compara métricas descriptivas sobre test. No seleccionar ni ajustar umbrales usando test. La Fase 3 abordará umbrales, significancia, comparación A/B y costos.
- Los datos son históricos de Taiwán; no se extrapolan automáticamente a México.